<h1>LD Pipeline 2024 (Version 1.0, PROD)</h1>

<h3>Lade die Umgebung</h3>

In [1]:
import os
import requests

try:
    initialized
except NameError:
    initialized = False

if not initialized:
    initialized = True
    os.chdir('../')
    import main
    from pipeline.base import Utils
    Utils.is_jupyter_notebook = True
    env = main.Env.int

In [2]:
main.step(name="generateViews", env=env)

2024-06-21 17:21:35 - Start building ld-views
2024-06-21 17:21:40 - Start building ld-view BEV324OD3240
2024-06-21 17:21:40 - Written ld-view BEV324OD3240
2024-06-21 17:21:40 - Start building ld-view BAU502OD5022
2024-06-21 17:21:41 - Written ld-view BAU502OD5022
2024-06-21 17:21:41 - Start building ld-view BEV324OD3241
2024-06-21 17:21:41 - Written ld-view BEV324OD3241
2024-06-21 17:21:41 - Start building ld-view BEV324OD3242
2024-06-21 17:21:41 - Written ld-view BEV324OD3242
2024-06-21 17:21:41 - Start building ld-view BEV324OD3243
2024-06-21 17:21:41 - Written ld-view BEV324OD3243
2024-06-21 17:21:41 - Start building ld-view BAU515OD5151
2024-06-21 17:21:41 - Written ld-view BAU515OD5151
2024-06-21 17:21:41 - Start building ld-view BAU512OD5121
2024-06-21 17:21:41 - Written ld-view BAU512OD5121
2024-06-21 17:21:41 - Start building ld-view BAU512OD5122
2024-06-21 17:21:41 - Written ld-view BAU512OD5122
2024-06-21 17:21:41 - Start building ld-view BAU513OD5131
2024-06-21 17:21:41 - Wr

<h3>Überprüfe, ob ein Start-Signal vorhanden ist</h3>

In [ ]:
utils = Utils()
utils.check_start_signal(env)

In [ ]:
utils.set_finish_signal(env)

<h3>Aktualisiere die Pipe-Tabellen</h3>

In [ ]:
main.step(name="copyHDBToPipeTables", env=env)

<h3>Generiere die Triple-Dateien</h3>

In [ ]:
names = [ 'code', 'cube', 'groupCode', 'hierarchy', 'legalFoundation',
         'measureUnit', 'measure', 'property', 'room', 'time', 'observation' ]

names = ['cube']

options = {
    'db_batch_size': 100000,
    'write_batch_size': 600000,
    'max_iteration': None
}

'''
options = {
    'db_batch_size': 1,
    'write_batch_size': 1,
    'max_iteration': 1
}
'''

for name in names:
    main.step(name=f"{name}Templating", env=env, options=options)

<h3>Setze den Fuseki-Server zurück (Optional)</h3>
<p>
    Der Fuseki-Server ist hier <a href="http://szhm58606:8080/fuseki#/dataset/ssz/query">hier</a>.
</p>

In [ ]:
def reset_fuseki_server():
    server_url = 'http://localhost:8080/fuseki/ssz/update'
    query = """
        DELETE WHERE {
            ?s ?p ?o.
        }
    """
    response = requests.post(server_url, data={'update': query})
    if response.status_code == 200:
        print("All triples are successfully deleted.")
    else:
        print(f"Status code: {response.status_code}, Error: {response.text}")

reset_fuseki_server()

<h3>Übertrage die Dateien in der Inbox zu dem lokalen Fuseki-Server</h3>
<p>
    Der Fuseki-Server ist hier <a href="http://szhm58606:8080/fuseki#/dataset/ssz/query">hier</a>.
</p>

In [ ]:
main.step(name="uploadToFuseki", env=env)

<h3>Übertrage die Dateien in der Inbox zu Stardog</h3>

In [ ]:
main.step(name="uploadToStardog", env=env)

<h3>Sende SPARQL-Abfragen an Stardog</h3>

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)

utils = Utils()
stardog_graph_uri = utils.get_stardog_graph_uri(env=env)
df = utils.execute_sparql(f"""
    SELECT * FROM <{stardog_graph_uri}> WHERE {{
        <https://ld.stadt-zuerich.ch/statistics/800299> ?pred ?obj
    }} LIMIT 100
""", env=env)
display(df)